# OfficeQA Retrieval Runner

This notebook is a thin Colab wrapper around the Python package in this repository. It now also includes post-run analysis cells that summarize metrics, export CSVs, and help inspect qualitative examples.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/hungngo2/cs445-final-project-retrieval.git officeqa-retrieval
%cd /content/officeqa-retrieval
!python -m pip install -e '.[ml,dev]'

In [ ]:
QUESTIONS_CSV = '/content/drive/MyDrive/officeqa/officeqa_pro.csv'
PDF_DIR = '/content/drive/MyDrive/officeqa/pdfs'
ARTIFACT_DIR = '/content/drive/MyDrive/officeqa/artifacts'
RESULTS_DIR = '/content/drive/MyDrive/officeqa/results'

!python scripts/download_officeqa_pdfs.py --questions-csv "$QUESTIONS_CSV" --pdf-dir "$PDF_DIR" --skip-existing
!python scripts/prepare_data.py --questions-csv "$QUESTIONS_CSV" --pdf-dir "$PDF_DIR" --manifest-out "$ARTIFACT_DIR/page_manifest.jsonl" --sanity-out "$ARTIFACT_DIR/sanity_questions.json"
!python scripts/build_page_index.py --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index-out "$ARTIFACT_DIR/page_bm25.pkl"

In [ ]:
!python scripts/run_retrieval_eval.py --questions-csv "$QUESTIONS_CSV" --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index "$ARTIFACT_DIR/page_bm25.pkl" --out-dir "$RESULTS_DIR/bm25" --model bm25 --shortlist-k 50
!python scripts/run_retrieval_eval.py --questions-csv "$QUESTIONS_CSV" --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index "$ARTIFACT_DIR/page_bm25.pkl" --out-dir "$RESULTS_DIR/clip_full" --model clip --crop-mode full --render-cache "$ARTIFACT_DIR/render_cache" --shortlist-k 50
!python scripts/run_retrieval_eval.py --questions-csv "$QUESTIONS_CSV" --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index "$ARTIFACT_DIR/page_bm25.pkl" --out-dir "$RESULTS_DIR/clip_fixed" --model clip --crop-mode fixed_2x2 --render-cache "$ARTIFACT_DIR/render_cache" --shortlist-k 50
!python scripts/run_retrieval_eval.py --questions-csv "$QUESTIONS_CSV" --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index "$ARTIFACT_DIR/page_bm25.pkl" --out-dir "$RESULTS_DIR/siglip_full" --model siglip --crop-mode full --render-cache "$ARTIFACT_DIR/render_cache" --shortlist-k 50
!python scripts/run_retrieval_eval.py --questions-csv "$QUESTIONS_CSV" --manifest "$ARTIFACT_DIR/page_manifest.jsonl" --index "$ARTIFACT_DIR/page_bm25.pkl" --out-dir "$RESULTS_DIR/siglip_fixed" --model siglip --crop-mode fixed_2x2 --render-cache "$ARTIFACT_DIR/render_cache" --shortlist-k 50
!python scripts/make_report_tables.py --results-dir "$RESULTS_DIR" --summary-out "$RESULTS_DIR/summary.csv"

## Post-Run Analysis

These cells load the persisted run artifacts from Google Drive. The experiment commands already save `metrics.json`, `ranked_pages.jsonl`, and `summary.csv`; the cells below turn those into plots and paper-friendly CSVs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

summary_path = Path(RESULTS_DIR) / 'summary.csv'
summary_df = pd.read_csv(summary_path).sort_values('page_mrr', ascending=False)
display(summary_df)

analysis_dir = Path(RESULTS_DIR) / 'analysis'
analysis_dir.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(analysis_dir / 'summary_table.csv', index=False)

plot_df = summary_df.set_index('run_name')[[
    'page_mrr',
    'page_recall_at_1',
    'page_recall_at_5',
    'page_recall_at_10',
    'doc_recall_at_1',
]]
ax = plot_df.plot(kind='bar', figsize=(12, 5), rot=30, title='OfficeQA Retrieval Summary by Run')
ax.set_ylabel('score')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(analysis_dir / 'summary_metrics.png', dpi=200, bbox_inches='tight')
plt.show()

print(f'Saved summary CSV to {analysis_dir / "summary_table.csv"}')
print(f'Saved summary plot to {analysis_dir / "summary_metrics.png"}')

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

records = []
for metrics_path in sorted(Path(RESULTS_DIR).glob('*/metrics.json')):
    data = json.loads(metrics_path.read_text())
    run_name = metrics_path.parent.name
    for row in data['per_query']:
        item = dict(row)
        item['run_name'] = run_name
        records.append(item)

per_query_df = pd.DataFrame(records)
per_query_df.to_csv(analysis_dir / 'per_query_metrics.csv', index=False)
display(per_query_df.head(20))

pivot = per_query_df.pivot(index='uid', columns='run_name', values='page_mrr').fillna(0).sort_index()
display(pivot.head(20))

if len(pivot) <= 30:
    ax = pivot.plot(kind='bar', figsize=(14, 6), rot=45, title='Per-question Page MRR by Run')
    ax.set_ylabel('page MRR')
    plt.tight_layout()
    plt.savefig(analysis_dir / 'per_query_page_mrr.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('Skipping per-question bar chart because there are more than 30 questions. Use the CSV export instead.')

difficulty_summary = (
    per_query_df.groupby(['run_name', 'difficulty'])[['page_mrr', 'doc_mrr', 'page_recall_at_10']]
    .mean()
    .reset_index()
)
difficulty_summary.to_csv(analysis_dir / 'difficulty_summary.csv', index=False)
display(difficulty_summary)

print(f'Saved per-query CSV to {analysis_dir / "per_query_metrics.csv"}')
print(f'Saved difficulty summary CSV to {analysis_dir / "difficulty_summary.csv"}')

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

TARGET_UID = 'UID0001'  # change this to inspect a different question

questions_df = pd.read_csv(QUESTIONS_CSV)
question_row = questions_df.loc[questions_df['uid'] == TARGET_UID].iloc[0]
print('Question UID:', TARGET_UID)
print('Question:', question_row['question'])
print('Gold source_files:', question_row['source_files'])
print('Gold source_docs:', question_row['source_docs'])

rows = []
for ranked_path in sorted(Path(RESULTS_DIR).glob('*/ranked_pages.jsonl')):
    run_name = ranked_path.parent.name
    with ranked_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            row = json.loads(line)
            if row['uid'] == TARGET_UID and row['rank'] <= 10:
                row['run_name'] = run_name
                rows.append(row)

qual_df = pd.DataFrame(rows).sort_values(['run_name', 'rank'])
qual_df.to_csv(analysis_dir / f'qualitative_{TARGET_UID}.csv', index=False)
display(qual_df[['run_name', 'rank', 'doc_id', 'page_num', 'score', 'bm25_score', 'matched_crop']])

print(f'Saved qualitative CSV to {analysis_dir / f"qualitative_{TARGET_UID}.csv"}')